Good Resource of Examples: https://cookbook.openai.com/

In [ ]:
import os
from openai import OpenAI

# Set API key (replace 'your-api-key-here' with your actual OpenAI API key)
# Option 1: Set it directly here (for quick testing)
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Option 2: Or export it in terminal before running: export OPENAI_API_KEY="your-api-key-here"

# Get API key from environment variable
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [52]:
# Let's make sure we are connected by using a model
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Hi, Aman here, how are you?"}]
)

print(response.choices[0].message.content)

Hello Aman! I'm an AI and don't have feelings, but I'm working perfectly and ready to assist you. How can I help you today?


In [53]:
## Create a new Eval for our problem
eval = client.evals.create(
    name="Medical Question Eval - GPT 5 Chat Latest",
    data_source_config={ # defines the schema of the eval dataset and what data will be available
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string"
                },
                "answer_idx": {
                    "type": "string"
                }
            },
            "required": [ "message", "answer_idx" ]
        },
        "include_sample_schema": True
    },
    testing_criteria=[ #  this will decide what is pass/fail
        {
            "type": "string_check",
            "name": "Checks that the predicted label matches actual",
            "operation": "eq",
            "input": "{{ sample.output_text }}",
            "reference": "{{ item.answer_idx }}" # the answer we expect from the model in our dataset
        }
    ]
)

# Store eval ID dynamically for use in subsequent cells
EVAL_ID = eval.id
print(f"EVAL_ID: {EVAL_ID}")

eval

EVAL_ID: eval_696b8bd6f0808191bc01c21f6f165cb7


EvalCreateResponse(id='eval_696b8bd6f0808191bc01c21f6f165cb7', created_at=1768655830, data_source_config=EvalCustomDataSourceConfig(schema_={'type': 'object', 'properties': {'item': {'type': 'object', 'properties': {'message': {'type': 'string'}, 'answer_idx': {'type': 'string'}}, 'required': ['message', 'answer_idx']}, 'sample': {'type': 'object', 'properties': {'model': {'type': 'string'}, 'choices': {'type': 'array', 'items': {'type': 'object', 'properties': {'message': {'type': 'object', 'properties': {'role': {'type': 'string', 'enum': ['assistant']}, 'content': {'type': ['string', 'array', 'null']}, 'refusal': {'type': ['boolean', 'null']}, 'tool_calls': {'type': ['array', 'null'], 'items': {'type': 'object', 'properties': {'type': {'type': 'string', 'enum': ['function']}, 'function': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'arguments': {'type': 'string'}}, 'required': ['name', 'arguments']}, 'id': {'type': 'string'}}, 'required': ['type', 'function', 'id']}

In [54]:
# Next: Upload a file programatically OR upload it through the UI on platform.openai.com
with open("medical_questions.jsonl", "rb") as f:
    file = client.files.create(
        file=f,
        purpose="evals"
    )

# Store file ID dynamically (or use environment variable as fallback)
DATASET_FILE_ID = file.id if file and hasattr(file, 'id') else os.getenv("DATASET_FILE_ID")
print(f"Using DATASET_FILE_ID: {DATASET_FILE_ID}")

file # keep track of the id

Using DATASET_FILE_ID: file-MkLwLwgHD98kcrtjg2vdpg


FileObject(id='file-MkLwLwgHD98kcrtjg2vdpg', bytes=15733, created_at=1768655835, filename='medical_questions.jsonl', object='file', purpose='evals', status='processed', expires_at=None, status_details=None)

In [55]:
# Next Step: Create a Run off of the Eval we defined.
# in other words: provide the data and specify the model we will use

# Use dynamic IDs: from eval/file objects (if available), or environment variables, or fallback to None
# If variables aren't set from previous cells, try to get them from environment variables
if 'EVAL_ID' not in locals() or not EVAL_ID:
    EVAL_ID = os.getenv("EVAL_ID")
if 'DATASET_FILE_ID' not in locals() or not DATASET_FILE_ID:
    DATASET_FILE_ID = os.getenv("DATASET_FILE_ID")

if not EVAL_ID or not DATASET_FILE_ID:
    raise ValueError(f"EVAL_ID and DATASET_FILE_ID must be set. "
                     f"Current values: EVAL_ID={EVAL_ID}, DATASET_FILE_ID={DATASET_FILE_ID}. "
                     f"Set them via environment variables or ensure cells 3 and 4 have been executed.")

print(f"Creating run with EVAL_ID: {EVAL_ID}, DATASET_FILE_ID: {DATASET_FILE_ID}")

run = client.evals.runs.create(
    eval_id = EVAL_ID,
    data_source={
        "type": "completions",
        "model": "gpt-5-chat-latest",
        "sampling_params": {
            "temperature": 0, # deterministic instead of random
            "max_completions_tokens": 1,
        },
        "source": {"type": "file_id", "id": DATASET_FILE_ID},
        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": "Given the following question and options, respond with only A or B."},
                {"role": "user", "content": "{{ item.message }}"},
            ],
        },
    },
)
run

Creating run with EVAL_ID: eval_696b8bd6f0808191bc01c21f6f165cb7, DATASET_FILE_ID: file-MkLwLwgHD98kcrtjg2vdpg


RunCreateResponse(id='evalrun_696b8be4c7648191afab6253fd01ba9d', created_at=1768655845, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-MkLwLwgHD98kcrtjg2vdpg', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='Given the following question and options, respond with only A or B.', role='developer', type='message'), EasyInputMessage(content='{{ item.message }}', role='user', type='message')], type='template'), model='gpt-5-chat-latest', sampling_params=SamplingParams(max_completion_tokens=None, reasoning_effort=None, response_format=None, seed=None, temperature=0.0, tools=None, top_p=None, max_completions_tokens=1), provider_credentials=None, modalities=None), error=None, eval_id='eval_696b8bd6f0808191bc01c21f6f165cb7', metadata={}, model='gpt-5-chat-latest', name=None, object='eval.run', per_model_usage=None, per_testing_criteria_results=None, report_url='https://platform.openai.com/evaluations/eval_

In [ ]:
run.report_url

RunCreateResponse(id='evalrun_696b8be4c7648191afab6253fd01ba9d', created_at=1768655845, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-MkLwLwgHD98kcrtjg2vdpg', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='Given the following question and options, respond with only A or B.', role='developer', type='message'), EasyInputMessage(content='{{ item.message }}', role='user', type='message')], type='template'), model='gpt-5-chat-latest', sampling_params=SamplingParams(max_completion_tokens=None, reasoning_effort=None, response_format=None, seed=None, temperature=0.0, tools=None, top_p=None, max_completions_tokens=1), provider_credentials=None, modalities=None), error=None, eval_id='eval_696b8bd6f0808191bc01c21f6f165cb7', metadata={}, model='gpt-5-chat-latest', name=None, object='eval.run', per_model_usage=None, per_testing_criteria_results=None, report_url='https://platform.openai.com/evaluations/eval_

In [57]:
# get the run
# Ensure EVAL_ID is available (from previous cells or environment)
if 'EVAL_ID' not in locals() or not EVAL_ID:
    EVAL_ID = os.getenv("EVAL_ID")
    if not EVAL_ID:
        raise ValueError("EVAL_ID must be set. Ensure cell 3 or 5 has been executed, or set EVAL_ID environment variable.")

run = client.evals.runs.retrieve(eval_id=EVAL_ID, run_id=run.id)
run.result_counts

ResultCounts(errored=0, failed=3, passed=38, total=41)